# DiscoveryStack SEO/GEO 101：Google Colab 本地多任務訓練

本 notebook 只接受 owner-approved immutable manifest 匯出的真實 JSONL。它會以固定 manifest、資料集摘要、split 與資料量 fail-closed 驗證資料，不會建立假資料、重新隨機切分或放寬 PII gate。

> 此實驗為 **101 筆 development proof of concept**。九個多任務頭的結果必須如實報告，不可宣稱是 production model。


In [ ]:
!pip -q install transformers scikit-learn sentencepiece
!nvidia-smi


## 私有資料載入備援

Google Drive 掛載的 OAuth 回傳若失敗，本 notebook 只接受從受控本機工作區直接傳入 Colab runtime 的同名私有 JSONL。該檔案不會公開分享、不會提交至 Git，並在下一儲存格以 immutable manifest 與 digest fail-closed 驗證。


In [ ]:
from google.colab import files
from pathlib import Path

EXPECTED_FILENAME = 'discoverystack-manifest-1-5c8917f1c3aa.jsonl'
uploaded = files.upload()
assert list(uploaded) == [EXPECTED_FILENAME], f'FAIL-CLOSED: expected exactly {EXPECTED_FILENAME}, got {list(uploaded)}'
DATA_PATH = Path('/content') / EXPECTED_FILENAME
assert DATA_PATH.exists(), f'FAIL-CLOSED: uploaded data file not found at {DATA_PATH}'
assert DATA_PATH.stat().st_size > 100_000, f'FAIL-CLOSED: uploaded data file too small ({DATA_PATH.stat().st_size} bytes)'
DATA_LOAD_METHOD = 'private_direct_colab_runtime_upload_drive_oauth_400'
print({'privateRuntimeUpload': True, 'filename': DATA_PATH.name, 'bytes': DATA_PATH.stat().st_size, 'dataLoadMethod': DATA_LOAD_METHOD})


In [ ]:
import hashlib
import json
import random
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

EXPECTED_MANIFEST_HASH = '5c8917f1c3aa9908c4af9b8216de0f056139f7aa4ae3756bcf29af7fcbb6bbdc'
EXPECTED_DATASET_DIGEST = 'c0cd6029382b5c9aba6baa1d348efa807758821889ce8e7b1f023ac100794569'
assert 'DATA_PATH' in globals() and 'DATA_LOAD_METHOD' in globals(), 'FAIL-CLOSED: private runtime upload cell has not run'
REQUIRED_STAGES = {'discovery', 'understanding', 'response', 'progression', 'conversion'}
SPLITS = {'train', 'validation', 'test'}
TASKS = ['journeyStage', 'searchIntents', 'contentTypes', 'audienceRoles', 'geoSignals', 'citationReadiness', 'technicalSeoSignals', 'frictionSignals', 'actionPriority']

assert DATA_PATH.exists(), f'FAIL-CLOSED: data file not found at {DATA_PATH}'
assert DATA_PATH.stat().st_size > 100_000, f'FAIL-CLOSED: data file too small ({DATA_PATH.stat().st_size} bytes)'
raw_lines = [line.rstrip('\n') for line in DATA_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
rows = [json.loads(line) for line in raw_lines]
DATASET_DIGEST = hashlib.sha256('\n'.join(raw_lines).encode('utf-8')).hexdigest()

assert len(rows) == 101, f'FAIL-CLOSED: expected 101 rows, found {len(rows)}'
assert DATASET_DIGEST == EXPECTED_DATASET_DIGEST, 'FAIL-CLOSED: dataset digest does not match frozen snapshot'
assert len({int(row['id']) for row in rows}) == 101, 'FAIL-CLOSED: duplicate artifact IDs'
assert {row.get('manifestHash') for row in rows} == {EXPECTED_MANIFEST_HASH}, 'FAIL-CLOSED: manifest hash mismatch'
assert {row.get('split') for row in rows} == SPLITS, 'FAIL-CLOSED: split values differ from immutable manifest'
assert Counter(row['split'] for row in rows) == Counter({'train': 74, 'validation': 14, 'test': 13}), 'FAIL-CLOSED: immutable split counts changed'
for index, row in enumerate(rows):
    assert isinstance(row.get('trainingText'), str) and row['trainingText'], f'FAIL-CLOSED: missing training text at row {index}'
    assert isinstance(row.get('targets'), dict), f'FAIL-CLOSED: missing targets at row {index}'
    missing = [task for task in TASKS if task not in row['targets']]
    assert not missing, f'FAIL-CLOSED: missing task targets {missing} at row {index}'
stages = Counter(row['targets']['journeyStage'] for row in rows)
assert set(stages) == REQUIRED_STAGES and min(stages.values()) >= 10, f'FAIL-CLOSED: invalid journey-stage distribution {dict(stages)}'

print('FAIL-CLOSED validation passed')
print({'datasetPath': str(DATA_PATH), 'bytes': DATA_PATH.stat().st_size, 'rows': len(rows), 'manifestHash': EXPECTED_MANIFEST_HASH, 'datasetDigest': DATASET_DIGEST, 'splits': dict(Counter(row['split'] for row in rows)), 'journeyStages': dict(stages)})


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer

MODEL_ID = 'distilbert-base-multilingual-cased'
SEED = 20260820
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
THRESHOLD = 0.5

assert torch.cuda.is_available(), 'FAIL-CLOSED: a CUDA GPU is required for this Colab training run'
DEVICE = torch.device('cuda')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def label_values(targets, task):
    value = targets[task]
    return [str(item) for item in value] if isinstance(value, list) else [str(value)]

label_maps = {}
for task in TASKS:
    vocabulary = sorted({label for row in rows for label in label_values(row['targets'], task)})
    assert vocabulary, f'FAIL-CLOSED: empty vocabulary for {task}'
    label_maps[task] = {label: index for index, label in enumerate(vocabulary)}

def encode(row):
    item = {'id': int(row['id']), 'split': row['split'], 'text': row['trainingText']}
    for task in TASKS:
        item[task] = [label_maps[task][label] for label in label_values(row['targets'], task)]
    return item

encoded_rows = [encode(row) for row in rows]
train_rows = [row for row in encoded_rows if row['split'] == 'train']
validation_rows = [row for row in encoded_rows if row['split'] == 'validation']
test_rows = [row for row in encoded_rows if row['split'] == 'test']
assert (len(train_rows), len(validation_rows), len(test_rows)) == (74, 14, 13), 'FAIL-CLOSED: split assignment changed after encoding'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print({'device': torch.cuda.get_device_name(0), 'labelCardinality': {task: len(mapping) for task, mapping in label_maps.items()}})


In [ ]:
def collate(batch):
    tokenized = tokenizer([item['text'] for item in batch], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
    labels = {}
    for task in TASKS:
        target = torch.zeros((len(batch), len(label_maps[task])), dtype=torch.float32)
        for row_index, item in enumerate(batch):
            target[row_index, item[task]] = 1.0
        labels[task] = target
    tokenized['labels'] = labels
    tokenized['ids'] = [item['id'] for item in batch]
    return tokenized

class MultiTaskModel(nn.Module):
    def __init__(self, model_id, task_label_maps):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_id)
        hidden_size = self.encoder.config.hidden_size
        self.heads = nn.ModuleDict({task: nn.Linear(hidden_size, len(task_label_maps[task])) for task in TASKS})

    def forward(self, input_ids, attention_mask):
        pooled = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return {task: self.heads[task](pooled) for task in TASKS}

def move_to_device(batch):
    return {
        'input_ids': batch['input_ids'].to(DEVICE),
        'attention_mask': batch['attention_mask'].to(DEVICE),
        'labels': {task: values.to(DEVICE) for task, values in batch['labels'].items()},
        'ids': batch['ids'],
    }

train_loader = DataLoader(train_rows, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, generator=torch.Generator().manual_seed(SEED))
validation_loader = DataLoader(validation_rows, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_rows, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
model = MultiTaskModel(MODEL_ID, label_maps).to(DEVICE)

# Smoke test uses five non-training examples and verifies every requested head before GPU training begins.
smoke_batch = move_to_device(collate((validation_rows + test_rows)[:5]))
with torch.no_grad():
    smoke_logits = model(smoke_batch['input_ids'], smoke_batch['attention_mask'])
assert set(smoke_logits) == set(TASKS)
assert all(smoke_logits[task].shape == smoke_batch['labels'][task].shape for task in TASKS)
SMOKE_TEST = {'nonTrainingExampleCount': 5, 'passed': True, 'taskHeads': TASKS}
print('non-training five-example smoke test passed')


In [ ]:
def compute_loss(logits, labels):
    task_losses = [nn.functional.binary_cross_entropy_with_logits(logits[task], labels[task]) for task in TASKS]
    return torch.stack(task_losses).mean()

def evaluate(loader, include_predictions=False):
    model.eval()
    total_loss, batches = 0.0, 0
    actual = {task: [] for task in TASKS}
    predicted = {task: [] for task in TASKS}
    prediction_rows = []
    with torch.no_grad():
        for batch in loader:
            batch = move_to_device(batch)
            logits = model(batch['input_ids'], batch['attention_mask'])
            total_loss += compute_loss(logits, batch['labels']).item()
            batches += 1
            for task in TASKS:
                y = batch['labels'][task].detach().cpu().numpy().astype(int)
                p = (torch.sigmoid(logits[task]).detach().cpu().numpy() >= THRESHOLD).astype(int)
                actual[task].append(y); predicted[task].append(p)
            if include_predictions:
                for position, artifact_id in enumerate(batch['ids']):
                    prediction_rows.append({'id': artifact_id, 'predicted': {task: np.flatnonzero((torch.sigmoid(logits[task][position]).detach().cpu().numpy() >= THRESHOLD)).tolist() for task in TASKS}})
    metrics = {'loss': total_loss / max(batches, 1)}
    for task in TASKS:
        y = np.concatenate(actual[task], axis=0)
        p = np.concatenate(predicted[task], axis=0)
        metrics[f'{task}_macro_f1'] = float(f1_score(y, p, average='macro', zero_division=0))
        metrics[f'{task}_micro_f1'] = float(f1_score(y, p, average='micro', zero_division=0))
    return metrics, prediction_rows

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=True)
started_at = datetime.now(timezone.utc)
history = []
for epoch in range(1, EPOCHS + 1):
    model.train(); total_loss = 0.0
    for batch in train_loader:
        batch = move_to_device(batch)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=True):
            logits = model(batch['input_ids'], batch['attention_mask'])
            loss = compute_loss(logits, batch['labels'])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
    validation_metrics, _ = evaluate(validation_loader)
    epoch_record = {'epoch': epoch, 'train_loss': total_loss / max(len(train_loader), 1), 'validation': validation_metrics}
    history.append(epoch_record)
    print(json.dumps(epoch_record, ensure_ascii=False))
completed_at = datetime.now(timezone.utc)
test_metrics, test_predictions = evaluate(test_loader, include_predictions=True)
metrics = {'validation': history[-1]['validation'], 'test': test_metrics, 'epochHistory': history}
print(json.dumps({'provider': 'google_colab_local', 'validation': metrics['validation'], 'test': metrics['test']}, ensure_ascii=False, indent=2))


In [ ]:
import shutil
import textwrap
from google.colab import files

run_id = f"manifest-1-{EXPECTED_MANIFEST_HASH[:12]}-{started_at.strftime('%Y%m%dT%H%M%SZ')}"
artifact_dir = Path('/content/DiscoveryStack_training_artifacts') / run_id
checkpoint_dir = artifact_dir / 'checkpoint'
checkpoint_dir.mkdir(parents=True, exist_ok=False)

state_dict_path = checkpoint_dir / 'model_state_dict.pt'
torch.save(model.state_dict(), state_dict_path)
tokenizer.save_pretrained(checkpoint_dir)
checkpoint_sha256 = hashlib.sha256(state_dict_path.read_bytes()).hexdigest()
model_definition = """import torch.nn as nn
from transformers import AutoModel

class MultiTaskModel(nn.Module):
    def __init__(self, model_id, task_label_maps, tasks):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_id)
        hidden_size = self.encoder.config.hidden_size
        self.heads = nn.ModuleDict({task: nn.Linear(hidden_size, len(task_label_maps[task])) for task in tasks})

    def forward(self, input_ids, attention_mask):
        pooled = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return {task: self.heads[task](pooled) for task in self.heads}
"""
(checkpoint_dir / 'model_definition.py').write_text(textwrap.dedent(model_definition), encoding='utf-8')

config = {
    'provider': 'google_colab_local', 'model': MODEL_ID, 'seed': SEED, 'maxLength': MAX_LENGTH,
    'batchSize': BATCH_SIZE, 'epochs': EPOCHS, 'learningRate': LEARNING_RATE, 'weightDecay': WEIGHT_DECAY,
    'threshold': THRESHOLD, 'taskHeads': TASKS, 'manifestHash': EXPECTED_MANIFEST_HASH,
    'datasetDigest': DATASET_DIGEST, 'exampleCount': len(rows), 'splitCounts': dict(Counter(row['split'] for row in rows)),
    'dataLoadMethod': DATA_LOAD_METHOD,
    'labelMaps': label_maps, 'device': torch.cuda.get_device_name(0), 'developmentOnly': True,
    'productionGate': {'minimumExamples': 150, 'minimumPerJourneyStage': 20, 'passed': False},
}
summary = {
    'provider': 'google_colab_local', 'manifestHash': EXPECTED_MANIFEST_HASH, 'datasetDigest': DATASET_DIGEST,
    'checkpointSha256': checkpoint_sha256, 'baseModelId': MODEL_ID, 'modelVersion': 'seo-geo-multitask-colab-v1',
    'metrics': metrics, 'smokeTest': SMOKE_TEST, 'startedAt': started_at.isoformat(), 'completedAt': completed_at.isoformat(),
    'dataLoadMethod': DATA_LOAD_METHOD,
    'artifactDirectory': str(artifact_dir), 'developmentOnly': True,
}
(artifact_dir / 'training-config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
(artifact_dir / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
(artifact_dir / 'run-summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
(artifact_dir / 'test-predictions.json').write_text(json.dumps(test_predictions, ensure_ascii=False, indent=2), encoding='utf-8')

archive_base = '/content/colab-training-artifacts'
archive_path = shutil.make_archive(archive_base, 'zip', root_dir=artifact_dir.parent, base_dir=artifact_dir.name)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print(f'Private Colab runtime artifacts: {artifact_dir}')
print(f'ZIP ready for download: {archive_path}')
files.download(archive_path)
